# Module 5 Lab — Fine-Grained Authorization for Agents

**Scenario:** Enterprise Procurement Agent

Build an authorization layer that combines:

```text
User authority
+
Agent/task authority
+
Resource relationships
+
Runtime attributes
+
Policy
=
ALLOW / DENY / ESCALATE
```

The core lab is local and deterministic. Optional sections connect to OpenFGA and OPA.

## Tools

- **Pydantic** — typed authorization requests/decisions
- **pandas** — policy matrices and decision evidence
- **OpenFGA SDK** — task/relationship authorization
- **OPA/Rego** — general policy decision point
- **Cedar** — policy examples and comparison
- **requests** — local PDP calls

The goal is to compare abstractions, not pretend one product fits every authorization problem.

In [ ]:
%pip install -q "pydantic>=2" pandas requests openfga_sdk
print("Dependencies installed.")

In [ ]:
from __future__ import annotations
from datetime import datetime, timezone, timedelta
from enum import Enum
from typing import Any, Optional
from uuid import uuid4
import os, json, time

import pandas as pd
import requests
from pydantic import BaseModel, Field

pd.set_option("display.max_colwidth",120)

## 1. Define the canonical authorization request

In [ ]:
class Decision(str,Enum):
    ALLOW="ALLOW"
    DENY="DENY"
    ESCALATE="ESCALATE"

class AuthorizationContext(BaseModel):
    amount: float=0
    vendor_id: Optional[str]=None
    vendor_approved: bool=False
    risk_score: float=0
    approval_state: str="not_required"
    country: str="CA"
    current_time: datetime=Field(default_factory=lambda:datetime.now(timezone.utc))

class AuthorizationRequest(BaseModel):
    request_id: str=Field(default_factory=lambda:f"auth-{uuid4().hex[:8]}")
    subject: str
    actor: str
    task_id: str
    action: str
    resource: str
    context: AuthorizationContext

class AuthorizationDecision(BaseModel):
    decision: Decision
    reason: str
    policy_ids: list[str]
    policy_version: str="authz-2026-08-1"
    expires_at: datetime=Field(default_factory=lambda:datetime.now(timezone.utc)+timedelta(seconds=30))
    evidence_id: str=Field(default_factory=lambda:f"evidence-{uuid4().hex[:8]}")

The LLM should not create trusted values such as `vendor_approved`.

Those fields should come from authoritative services.

## 2. Baseline RBAC

In [ ]:
ROLE_PERMISSIONS={
    "ProcurementManager":{"vendor:read","purchase_order:create"},
    "AnalyticsManager":{"vendor:read","purchase_order:request"},
    "ResearchAgent":{"vendor:read"},
}

USER_ROLES={
    "user:123":"AnalyticsManager",
    "user:proc-manager":"ProcurementManager",
}

def rbac_allowed(principal:str,action:str)->bool:
    role=USER_ROLES.get(principal)
    return action in ROLE_PERMISSIONS.get(role,set())

print(rbac_allowed("user:123","purchase_order:create"))
print(rbac_allowed("user:proc-manager","purchase_order:create"))

### RBAC limitation

The procurement manager role can create POs, but RBAC alone does not answer:

- which department,
- which task,
- which vendor,
- which amount,
- whether the task expired.

## 3. ABAC

In [ ]:
def abac_decision(req:AuthorizationRequest)->AuthorizationDecision:
    c=req.context

    if req.action!="purchase_order:create":
        return AuthorizationDecision(
            decision=Decision.DENY,
            reason="ABAC demo only permits purchase-order creation.",
            policy_ids=["abac-action"],
        )

    if not c.vendor_approved:
        return AuthorizationDecision(
            decision=Decision.DENY,
            reason="Vendor is not approved.",
            policy_ids=["abac-vendor"],
        )

    if c.amount>5000:
        return AuthorizationDecision(
            decision=Decision.ESCALATE,
            reason="Amount exceeds autonomous threshold.",
            policy_ids=["abac-amount"],
        )

    if c.risk_score>=0.7:
        return AuthorizationDecision(
            decision=Decision.ESCALATE,
            reason="Risk score requires human review.",
            policy_ids=["abac-risk"],
        )

    return AuthorizationDecision(
        decision=Decision.ALLOW,
        reason="Attributes satisfy policy.",
        policy_ids=["abac-default"],
    )

## 4. ReBAC / task relationships

In [ ]:
# Local relationship graph for teaching before live OpenFGA.
TASK_ASSIGNEE={
    "task:123":"agent:procurement-v1"
}
TASK_DELEGATOR={
    "task:123":"user:123"
}
TASK_RESOURCE={
    "task:123":"department:data-ai"
}
TASK_PERMISSIONS={
    "task:123":{"purchase_order:create","vendor:read"}
}
USER_RESOURCES={
    "user:123":{"department:data-ai"}
}

def rebac_task_allowed(req:AuthorizationRequest)->tuple[bool,str]:
    if TASK_ASSIGNEE.get(req.task_id)!=req.actor:
        return False,"Calling agent is not assigned to task."
    if TASK_DELEGATOR.get(req.task_id)!=req.subject:
        return False,"Task delegator mismatch."
    if TASK_RESOURCE.get(req.task_id)!=req.resource:
        return False,"Task does not grant access to resource."
    if req.action not in TASK_PERMISSIONS.get(req.task_id,set()):
        return False,"Task does not grant this action."
    if req.resource not in USER_RESOURCES.get(req.subject,set()):
        return False,"Original user lacks resource access."
    return True,"User + task + agent relationships are valid."

## 5. Dual authorization

In [ ]:
def combined_authorization(req:AuthorizationRequest)->AuthorizationDecision:
    relationship_ok,reason=rebac_task_allowed(req)
    if not relationship_ok:
        return AuthorizationDecision(
            decision=Decision.DENY,
            reason=reason,
            policy_ids=["rebac-task-binding"],
        )

    return abac_decision(req)

In [ ]:
safe=AuthorizationRequest(
    subject="user:123",
    actor="agent:procurement-v1",
    task_id="task:123",
    action="purchase_order:create",
    resource="department:data-ai",
    context=AuthorizationContext(
        amount=4500,
        vendor_id="vendor-acme",
        vendor_approved=True,
        risk_score=0.25,
    ),
)

combined_authorization(safe)

## 6. Negative tests

In [ ]:
cases=[
    ("safe",safe),
    ("wrong agent",safe.model_copy(update={"actor":"agent:other"})),
    ("wrong resource",safe.model_copy(update={"resource":"department:finance"})),
    ("bad vendor",safe.model_copy(update={"context":safe.context.model_copy(update={"vendor_approved":False})})),
    ("high amount",safe.model_copy(update={"context":safe.context.model_copy(update={"amount":9000})})),
    ("high risk",safe.model_copy(update={"context":safe.context.model_copy(update={"risk_score":0.85})})),
]

rows=[]
for name,req in cases:
    d=combined_authorization(req)
    rows.append({"case":name,"decision":d.decision.value,"reason":d.reason})
display(pd.DataFrame(rows))

## 7. Policy Enforcement Point

In [ ]:
AUTH_EVIDENCE=[]

def governed_tool_call(req:AuthorizationRequest):
    d=combined_authorization(req)

    record={
        "timestamp":datetime.now(timezone.utc).isoformat(),
        "request_id":req.request_id,
        "subject":req.subject,
        "actor":req.actor,
        "task_id":req.task_id,
        "action":req.action,
        "resource":req.resource,
        "context":req.context.model_dump(mode="json"),
        "decision":d.decision.value,
        "reason":d.reason,
        "policy_ids":d.policy_ids,
        "policy_version":d.policy_version,
        "evidence_id":d.evidence_id,
        "executed":False,
    }

    if d.decision==Decision.ALLOW:
        record["executed"]=True
        record["result"]={"po_id":f"PO-{uuid4().hex[:6]}","status":"created"}

    AUTH_EVIDENCE.append(record)
    return record

governed_tool_call(safe)

## 8. Fail closed

The policy enforcement point must define behavior when the PDP is unavailable.

For state-changing tools, a conservative default is:

```text
authorization unavailable → DENY or ESCALATE
```

not:

```text
authorization unavailable → execute anyway
```

In [ ]:
def external_pdp_simulation(available:bool,req:AuthorizationRequest):
    if not available:
        return AuthorizationDecision(
            decision=Decision.DENY,
            reason="Authorization service unavailable; fail closed.",
            policy_ids=["availability-fail-closed"],
        )
    return combined_authorization(req)

external_pdp_simulation(False,safe)

## 9. Authorization decision caching

In [ ]:
AUTH_CACHE={}

def cache_key(req:AuthorizationRequest):
    return (
        req.subject,req.actor,req.task_id,req.action,req.resource,
        req.context.vendor_id,req.context.amount,req.context.vendor_approved,
        round(req.context.risk_score,2),req.context.approval_state
    )

def cached_authorize(req:AuthorizationRequest,ttl_seconds:int=10):
    key=cache_key(req)
    now=time.time()

    if key in AUTH_CACHE:
        decision,created=AUTH_CACHE[key]
        if now-created<=ttl_seconds:
            return decision,"cache_hit"

    decision=combined_authorization(req)
    AUTH_CACHE[key]=(decision,now)
    return decision,"fresh"

print(cached_authorize(safe))
print(cached_authorize(safe))

### Caching warning

For high-impact actions, authorization should be evaluated close to execution.

Cache policy should consider:

- action sensitivity,
- policy changes,
- task revocation,
- resource changes,
- approval changes,
- maximum stale time.

## 10. TOCTOU — reauthorize at execution

In [ ]:
# Authorization is checked.
decision=combined_authorization(safe)
print("Initial:",decision.decision.value)

# Vendor becomes disallowed before execution.
changed=safe.model_copy(update={
    "context":safe.context.model_copy(update={"vendor_approved":False})
})

# Reauthorization catches the change.
print("At execution:",combined_authorization(changed).decision.value)

## 11. OpenFGA model for task-scoped tool authorization

The current OpenFGA agent guidance supports task grants, session/agent scoping, conditions, and agent binding.

Conceptual model:

In [ ]:
OPENFGA_MODEL = """
model
  schema 1.1

type user

type task

type agent
  relations
    define task: [task]

type department
  relations
    define member: [user]

type tool
  relations
    define calling_agent: [agent]
    define can_call: [task] and task from calling_agent
"""
print(OPENFGA_MODEL)

Conceptual tuples:

```text
task:123 relation task object agent:procurement-v1
task:123 relation can_call object tool:create_purchase_order
user:123 relation member object department:data-ai
```

At check time, supply contextual information that binds the calling agent and validates task access.

## 12. Optional OpenFGA SDK

In [ ]:
try:
    import openfga_sdk
    print("OpenFGA SDK available.")
except Exception as exc:
    print("OpenFGA import issue:",exc)

if os.getenv("FGA_API_URL") and os.getenv("FGA_STORE_ID"):
    print("Live OpenFGA configuration detected.")
    print("Use the official SDK examples for your installed version to create the model, write tuples, and call Check.")
else:
    print("Set FGA_API_URL / FGA_STORE_ID / FGA_MODEL_ID for the optional live extension.")

# 13. Cedar policy

Cedar expresses fine-grained authorization using principal, action, resource, and context.

In [ ]:
CEDAR_POLICY = """
permit (
  principal == Agent::"procurement-v1",
  action == Action::"CreatePurchaseOrder",
  resource
)
when {
  context.task == "task:123" &&
  context.amount <= 5000 &&
  context.vendorApproved == true &&
  context.riskScore < 0.7
};

forbid (
  principal,
  action == Action::"CreatePurchaseOrder",
  resource
)
when {
  context.sanctionedVendor == true
};
"""
print(CEDAR_POLICY)

### Why explicit forbid?

Critical restrictions should remain restrictions even if another permit policy also matches.

Always verify behavior against the current Cedar evaluator/service semantics used by your implementation.

## 14. OPA/Rego policy

In [ ]:
REGO_POLICY = """
package agentauthz

default decision := {
  "result": "DENY",
  "reason": "No allow rule matched."
}

decision := {
  "result": "DENY",
  "reason": "Task relationship invalid."
} if {
  input.relationship_valid == false
}

decision := {
  "result": "DENY",
  "reason": "Vendor is not approved."
} if {
  input.relationship_valid
  not input.context.vendor_approved
}

decision := {
  "result": "ESCALATE",
  "reason": "Amount requires approval."
} if {
  input.relationship_valid
  input.context.vendor_approved
  input.context.amount > 5000
}

decision := {
  "result": "ALLOW",
  "reason": "Task and attributes satisfy policy."
} if {
  input.relationship_valid
  input.context.vendor_approved
  input.context.amount <= 5000
  input.context.risk_score < 0.7
}
"""
with open("agent_authz.rego","w") as f:
    f.write(REGO_POLICY)
print(REGO_POLICY)

Run OPA locally:

```bash
docker run --rm -p 8181:8181   -v "$PWD:/policies"   openpolicyagent/opa:latest   run --server /policies/agent_authz.rego
```

In [ ]:
def opa_check(req:AuthorizationRequest):
    rel_ok,_=rebac_task_allowed(req)
    payload={
        "input":{
            "relationship_valid":rel_ok,
            "context":{
                "vendor_approved":req.context.vendor_approved,
                "amount":req.context.amount,
                "risk_score":req.context.risk_score,
            }
        }
    }
    r=requests.post(
        "http://localhost:8181/v1/data/agentauthz/decision",
        json=payload,
        timeout=3,
    )
    r.raise_for_status()
    return r.json()["result"]

try:
    print(opa_check(safe))
except Exception as exc:
    print("OPA is not running:",type(exc).__name__)

## 15. Compare policy engines conceptually

In [ ]:
comparison=pd.DataFrame([
    {
        "tool":"OpenFGA",
        "core_model":"relationships / tuples",
        "best_here":"user-task-agent-resource binding",
        "dynamic_context":"conditions/contextual tuples",
    },
    {
        "tool":"Cedar",
        "core_model":"principal-action-resource-context",
        "best_here":"fine-grained amount/vendor/risk policy",
        "dynamic_context":"first-class context",
    },
    {
        "tool":"OPA/Rego",
        "core_model":"declarative policy over structured data",
        "best_here":"cross-cutting enterprise workflow policy",
        "dynamic_context":"arbitrary JSON input",
    },
])
display(comparison)

## 16. Human approval is an input, not a master override

In [ ]:
approved=safe.model_copy(update={
    "context":safe.context.model_copy(update={
        "amount":9000,
        "approval_state":"manager_approved",
    })
})

def abac_with_approval(req:AuthorizationRequest):
    c=req.context

    if not c.vendor_approved:
        return AuthorizationDecision(
            decision=Decision.DENY,
            reason="Hard vendor restriction remains even with approval.",
            policy_ids=["vendor-hard-deny"],
        )

    if c.amount>5000 and c.approval_state!="manager_approved":
        return AuthorizationDecision(
            decision=Decision.ESCALATE,
            reason="Manager approval required.",
            policy_ids=["approval-threshold"],
        )

    return AuthorizationDecision(
        decision=Decision.ALLOW,
        reason="Context including required approval satisfies policy.",
        policy_ids=["approval-aware-policy"],
    )

print(abac_with_approval(approved))
print(abac_with_approval(approved.model_copy(update={
    "context":approved.context.model_copy(update={"vendor_approved":False})
})))

## 17. Authorization regression suite

In [ ]:
tests=[
    ("valid",safe,Decision.ALLOW),
    ("wrong agent",safe.model_copy(update={"actor":"agent:other"}),Decision.DENY),
    ("wrong resource",safe.model_copy(update={"resource":"department:finance"}),Decision.DENY),
    ("unapproved vendor",safe.model_copy(update={
        "context":safe.context.model_copy(update={"vendor_approved":False})
    }),Decision.DENY),
    ("above amount",safe.model_copy(update={
        "context":safe.context.model_copy(update={"amount":5001})
    }),Decision.ESCALATE),
]

results=[]
for name,req,expected in tests:
    d=combined_authorization(req)
    results.append({
        "test":name,
        "expected":expected.value,
        "actual":d.decision.value,
        "pass":d.decision==expected,
        "reason":d.reason,
    })

df=pd.DataFrame(results)
display(df)
assert df["pass"].all()

## 18. Evidence review

In [ ]:
display(pd.DataFrame(AUTH_EVIDENCE))

Record:

- subject,
- actor,
- task,
- resource,
- action,
- trusted context,
- policy IDs,
- policy version,
- decision,
- decision expiry,
- execution status.

This lets teams answer:

> Why did this agent have permission to perform that action?

# 19. State-of-the-art research review

Fine-grained authorization for autonomous agents is still evolving.

Three recent research directions are worth studying:

### AutoCedar

Verifier-guided synthesis of Cedar policies from natural-language requirements.

Lesson:

> Convert prose into a reviewed formal target, then verify generated policy against that target.

### Prose2Policy

Pipeline for generating executable Rego from natural-language access-control policies with validation/testing stages.

Lesson:

> Compilation is not enough; generated policy needs behavioral tests.

### FAVA

Research on permission-carrying, evidence-backed authorization graphs for dynamic agent execution.

Lesson:

> Static tool permissions may be insufficient when authorization depends on evolving runtime state and data flow.

These are useful research signals, not replacements for mature authorization infrastructure.

# 20. Exercises

### A — Add country restrictions

Only allow purchase orders in CA and EU.

### B — Add task expiry

Make task authorization fail after 30 minutes.

### C — Call count

Grant a task exactly two email sends.

### D — Session-level permission

Model “allow this tool for this session” in OpenFGA.

### E — Agent-level permission

Compare “always allow this agent” with task-level grants and document the risk difference.

### F — Policy outage

Implement:

- read-only fallback,
- hard deny for state changes.

### G — Verified Permissions

Use a real Cedar policy store and call `IsAuthorized`.

### H — AgentCore Policy

Map the PO tool into AgentCore Gateway and implement Cedar policies on tool arguments.

### I — Policy mutation tests

Create mutations:

- `<= 5000` → `< 5000`
- approved vendor check removed
- risk threshold changed

Confirm tests detect unintended access changes.

# 21. Key takeaways

1. OAuth scope is not task-level authorization.
2. Enterprise agents need user + task/agent + context checks.
3. RBAC, ABAC, ReBAC, and contextual policy complement each other.
4. OpenFGA is particularly strong for task/resource relationships.
5. Cedar is designed for principal/action/resource/context authorization.
6. OPA provides general policy evaluation over structured input.
7. Separate PDP and PEP.
8. Default deny for state-changing actions.
9. Use authoritative sources for security attributes.
10. Reauthorize near execution for high-impact actions.
11. Human approval is an input, not a universal override.
12. Treat policy code like application code: version, review, test, observe.
13. Experimental LLM policy synthesis needs formal/behavioral verification.